# Colab runner: train + inspect through the repo's CLI

Thin cells only. Every step calls `python -m src.cli.<tool>`, so the notebook never
drifts from the repo (test_run2's copied trainer lost `first_token_accuracy`).

- **Resume:** re-run the *same* cells with the same `RUN_NAME`; training continues from
  the Hub's `last-checkpoint/` and appends to the same W&B run.
- **No idle burn:** the run cell trains, evaluates, copies results to Drive and releases
  the runtime, so Colab never sits idle on a GPU.
- **Fresh code:** the repo is copied from Drive to local disk each session. Sync your
  Drive copy with the repo first (`outputs/` is skipped).

In [ ]:
# ---- What to run ----
LLM_CONFIG = "qlora/llama32_1b/seed43"  # configs/train_llm/<this>.yaml
RUN_NAME = "e7-llama32-1b-seed43"       # names the Hub repo + W&B run; keep it to resume
INSPECT_SPLITS = ["validation"]         # add "test" only for the final model (E15)
TRAIN = True                            # False = only inspect an existing adapter
TRAIN_ARGS = ""                         # e.g. "--max-train-samples 2000" or "--resume never"
INSPECT_ARGS = ""                       # e.g. "--base-only --few-shot 5" for baselines
RELEASE_RUNTIME_WHEN_DONE = True        # True for long runs; False while running several in one session

In [ ]:
# ---- Setup: Drive -> local disk, install ----
# Fail before the slow install if Colab gave us a CPU runtime
# (fix: Runtime -> Change runtime type -> L4 GPU, then run all cells again).
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 GPU"
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
DRIVE_REPO = "/content/drive/MyDrive/nba_bets"
# .gdoc files are Google Docs shortcuts, not files: rsync cannot read them.
!rsync -a --delete --exclude outputs --exclude .venv --exclude mlruns --exclude '*.gdoc' {DRIVE_REPO}/ /content/nba_bets/
%cd /content/nba_bets
!pip install -q .[gpu]
# Colab preinstalls torchao 0.10. peft >= 0.21 raises on any torchao < 0.16 when
# it wraps an unquantized model (bf16 runs). Nothing here uses torchao.
!pip uninstall -y -q torchao

In [ ]:
# ---- Secrets -> env (`!` subprocesses cannot read Colab Secrets themselves) ----
import os
from google.colab import userdata
for name in ("HF_llm_training_token", "WANDB_API_KEY"):
    os.environ[name] = userdata.get(name)

In [ ]:
# ---- Run: train, inspect, save to Drive, release ----
# Colab's `!` does not report a failed command back to Python, so a crashed
# training run used to fall through to inspect. run() streams the output live
# (progress bars included) and raises on a non-zero exit, stopping the cell.
# Pressing Stop interrupts only this notebook's Python, not the child process,
# so run() stops the child itself - otherwise training keeps going unseen.
import os, signal, subprocess, sys

def run(cmd):
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True
    )
    try:
        for chunk in iter(lambda: proc.stdout.read1(4096), b""):
            sys.stdout.write(chunk.decode(errors="replace"))
            sys.stdout.flush()
    except KeyboardInterrupt:
        os.killpg(proc.pid, signal.SIGINT)   # like Ctrl+C in a terminal
        try:
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        print(f"\nStopped: {cmd}", flush=True)
        raise
    if proc.wait() != 0:
        raise RuntimeError(f"Failed (exit code {proc.returncode}): {cmd}")



In [ ]:
CFG = f"configs/train_llm/{LLM_CONFIG}.yaml"
# A fresh or restarted runtime starts in /content: only the setup cell copies the
# repo to local disk, and `python -m src.cli...` fails anywhere else.
assert os.path.isdir("/content/nba_bets/src"), "Repo not on local disk: run the setup cell (or Run all)"
os.chdir("/content/nba_bets")
assert os.path.isfile(CFG), f"{CFG} not on Drive: run `make drive-sync ONLY=code` locally, then Run all"
try:
    if TRAIN:
        run(f"python -m src.cli.train_llm --config {CFG} --run-name {RUN_NAME} {TRAIN_ARGS}")
    for split in INSPECT_SPLITS:
        run(f"python -m src.cli.inspect_llm --config {CFG} --run-name {RUN_NAME} --split {split} --size 5000 {INSPECT_ARGS}")
    # Drive stays mounted from the setup cell for the whole session. Unmounting
    # after each run is what broke the next run's copy (it could not remount).
    # The run folder exists only if something was inspected; bench runs inspect nothing.
    # Layout: docs/reference/outputs.md (outputs/runs/<run>/ holds predictions, metrics, run.json, llm/).
    run(f"mkdir -p outputs/runs/{RUN_NAME}/tracking {DRIVE_REPO}/outputs/runs && cp -r outputs/runs/{RUN_NAME} {DRIVE_REPO}/outputs/runs/")
    run(f"cp mlflow.db {DRIVE_REPO}/outputs/runs/{RUN_NAME}/tracking/mlflow.db 2>/dev/null || true")
finally:
    if RELEASE_RUNTIME_WHEN_DONE:
        drive.flush_and_unmount()   # without this, recent Drive writes can be lost
        from google.colab import runtime
        runtime.unassign()          # disconnects and deletes the runtime; billing stops

In [ ]:
# ---- End of session (only when RELEASE_RUNTIME_WHEN_DONE was False) ----
# Drive uploads in the background; flushing makes sure the last run's copies
# reach Drive before you disconnect. Run it once, then disconnect the runtime.
drive.flush_and_unmount()